# ComfyUI en Google Colab con Google Drive
Este cuaderno ejecuta ComfyUI en Google Colab guardando modelos, nodos y salidas en Google Drive.

### ⚠️ Recomendaciones de Uso
- **GPU:** Asegúrate de activar una GPU (**Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**).
- **Almacenamiento:** Los modelos pesados ocupan varios GBs. Verifica tu espacio en Google Drive.
- **Cierre de sesión:** Al terminar la sesión, ve a *Entorno de ejecución > Desconectar y eliminar el entorno* para no consumir cuota de GPU.

In [ ]:
# Paso 1: Verificación de GPU y Montaje de Google Drive
import os
import torch
from google.colab import drive

print('Verificando disponibilidad de GPU...')
if not torch.cuda.is_available():
    print('❌ ALERTA: No hay GPU activa. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución y selecciona GPU T4.')
else:
    print(f'✅ GPU detectada: {torch.cuda.get_device_name(0)}')

print('\nMontando Google Drive...')
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/ComfyUI'
print(f'Ruta base de ComfyUI establecida en: {BASE_PATH}')

In [ ]:
# Paso 2: Crear o verificar estructura de carpetas persistentes
import os

BASE_PATH = '/content/drive/MyDrive/ComfyUI'
directories = [
    f'{BASE_PATH}/models/checkpoints',
    f'{BASE_PATH}/models/loras',
    f'{BASE_PATH}/models/vae',
    f'{BASE_PATH}/models/controlnet',
    f'{BASE_PATH}/models/clip',
    f'{BASE_PATH}/models/unet',
    f'{BASE_PATH}/custom_nodes',
    f'{BASE_PATH}/output'
]

for d in directories:
    os.makedirs(d, exist_ok=True)
print('✅ Estructura de carpetas verificada en Google Drive.')

In [ ]:
# Paso 3: Instalación de ComfyUI y ComfyUI-Manager
import os

BASE_PATH = '/content/drive/MyDrive/ComfyUI'

if not os.path.exists(f'{BASE_PATH}/main.py'):
    print('Clonando repositorio de ComfyUI en Google Drive...')
    !git clone https://github.com/comfyanonymous/ComfyUI {BASE_PATH}
else:
    print('✅ ComfyUI ya está instalado en Google Drive.')

%cd {BASE_PATH}
!pip install -r requirements.txt --quiet

# Instalar ComfyUI-Manager
MANAGER_PATH = f'{BASE_PATH}/custom_nodes/comfyui-manager'
if not os.path.exists(MANAGER_PATH):
    print('Instalando ComfyUI-Manager...')
    !git clone https://github.com/ltdrdata/ComfyUI-Manager {MANAGER_PATH}
else:
    print('✅ ComfyUI-Manager ya está instalado.')

In [ ]:
# Paso 4: Gestor interactivo de descargas (Formulario para Móvil / Android)
import os
import ipywidgets as widgets
from IPython.display import display
from google.colab import userdata

!apt-get install -y aria2 --quiet

provider_dropdown = widgets.Dropdown(options=['Hugging Face', 'Civitai', 'URL Directa'], value='Hugging Face', description='Proveedor:')
resource_type = widgets.Dropdown(options=['checkpoints', 'loras', 'vae', 'controlnet', 'unet'], value='checkpoints', description='Tipo:')
url_input = widgets.Text(placeholder='Pegar enlace de descarga aquí', description='URL:', layout=widgets.Layout(width='90%'))
filename_input = widgets.Text(placeholder='Ej: modelo.safetensors', description='Nombre:', layout=widgets.Layout(width='90%'))
download_button = widgets.Button(description='Iniciar Descarga', button_style='primary', icon='download')
output_area = widgets.Output()

def run_download(b):
    with output_area:
        output_area.clear_output()
        url = url_input.value.strip()
        name = filename_input.value.strip()
        target_folder = f'/content/drive/MyDrive/ComfyUI/models/{resource_type.value}'
        os.makedirs(target_folder, exist_ok=True)
        
        if not url or not name:
            print('❌ Por favor introduce la URL y el nombre del archivo.')
            return
            
        target_file = os.path.join(target_folder, name)
        if os.path.exists(target_file):
            print(f'⚠️ El archivo {name} ya existe en {target_folder}. Skipping...')
            return
            
        headers = ''
        if provider_dropdown.value == 'Hugging Face':
            try:
                hf_token = userdata.get('HF_TOKEN')
                if hf_token:
                    headers = f'--header="Authorization: Bearer {hf_token}"'
            except Exception:
                pass
        elif provider_dropdown.value == 'Civitai':
            try:
                civitai_token = userdata.get('CIVITAI_TOKEN')
                if civitai_token:
                    url += f'?token={civitai_token}'
            except Exception:
                pass
                
        print(f'Descargando {name} en {target_folder}...')
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {headers} "{url}" -d "{target_folder}" -o "{name}"'
        os.system(cmd)
        if os.path.exists(target_file):
            print(f'✅ Descarga completada: {name}')
        else:
            print('❌ Error durante la descarga.')

download_button.on_click(run_download)
display(widgets.VBox([provider_dropdown, resource_type, url_input, filename_input, download_button, output_area]))

In [ ]:
# Paso 5: Iniciar ComfyUI con Túnel de Cloudflare
import os

BASE_PATH = '/content/drive/MyDrive/ComfyUI'
%cd {BASE_PATH}

# Instalar Cloudflare Tunnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb --quiet
!dpkg -i cloudflared-linux-amd64.deb --quiet

print('🚀 Iniciando servidor ComfyUI y generando URL pública...')
!cloudflared tunnel --url http://127.0.0.1:8188 2>&1 | grep -o 'https://.*\.trycloudflare\.com' & python main.py --dont-print-server